# PWKD Option 2 — Prune ResNet50 Directly

Implements **Pruning While Knowledge Distillation** (Wang et al., 2025) using:
- **Teacher**: frozen pretrained ResNet50 (RadImageNet weights)
- **Student**: a deep copy of the *same* pretrained ResNet50, fine-tuned
  with PWKD simultaneously pruning and distilling

This is a same-architecture setup, closely analogous to the paper's
EDSR-32-256 → EDSR-16-64 setup but keeping the architecture fixed and
instead driving channels to zero through the differentiable sparsity penalty.
The wavelet channel-projection step is skipped (teacher/student channels match).

Starting from pretrained weights means the student begins at the teacher's
F1 level (~0.49) and PWKD nudges it toward a smaller, slightly lower-F1 model —
a much more favourable trade-off than training from scratch.

Produces 5 compressed models at pruning ratios [10%, 25%, 50%, 70%, 90%],
saved to `trained_models/pwkd_self_r50_<ratio>/` and uploaded to HuggingFace.

**Run all cells top to bottom. Requires GPU.**

In [2]:
import os, copy, subprocess
import torch

target = 'CS6423_knowledge_distillation_project'
if not os.getcwd().endswith(target):
    import sys
    os.chdir(os.path.join(os.getcwd(), target))
    if os.getcwd() not in sys.path:
        sys.path.insert(0, os.getcwd())

print(f'Working dir: {os.getcwd()}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

subprocess.run(['pip', 'install', 'PyWavelets', '--quiet'], check=True)

Working dir: /home/cor10/CS6423_knowledge_distillation_project
Device: cuda


CompletedProcess(args=['pip', 'install', 'PyWavelets', '--quiet'], returncode=0)

In [3]:
import pandas as pd
from modules.dataset_prepper import datasetPrepper

data_prep = datasetPrepper(
    dataframe_path='data/labels.csv',
    image_dir='data/test_images',
).prepare(compute_class_weights=True)

NUM_CLASSES = len(data_prep.class_names)
print(f'Classes: {NUM_CLASSES}')
print(f'Train batches: {len(data_prep.train_loader)} | Val batches: {len(data_prep.val_loader)}')

Classes: 61
Train batches: 249 | Val batches: 50


In [4]:
from modules.imagenet_loader import ImagenetLoader
from modules.evaluate_model import ModelEvaluator

loader = ImagenetLoader()

# Load pretrained ResNet50 — this serves as both the frozen teacher
# and the starting point for each student deep copy
resnet50_pretrained = loader.load_radimagenet_resnet50(
    weights_path='trained_models/resnet50_baseline_gpu_new/resnet50_baseline_gpu_new.pth',
    load_type='load'
)
resnet50_pretrained = resnet50_pretrained.to(device)

evaluator = ModelEvaluator(
    data_loader=data_prep.val_loader,
    class_names=data_prep.class_names,
    device=str(device),
)

baseline_metrics = evaluator.evaluate_single(resnet50_pretrained, 'ResNet50_baseline')
BASELINE_PARAMS  = baseline_metrics['total_parameters']
print(f'ResNet50 baseline F1:    {baseline_metrics["f1_macro"]:.4f}')
print(f'ResNet50 baseline params: {BASELINE_PARAMS:,}')


Warming up ResNet50_baseline...
Running inference...
ResNet50 baseline F1:    0.4867
ResNet50 baseline params: 23,633,021


## Same-architecture note

Because teacher and student are both ResNet50, their channel counts match at
every stage. The MVRM projection step (`teacher_proj`) becomes an identity
operation — `PWKDLoss` handles this automatically when `teacher_channels ==
student_channels`.

| Stage   | ResNet50 teacher | ResNet50 student |
|---------|-----------------|------------------|
| layer2  | 512             | 512              |
| layer3  | 1024            | 1024             |
| layer4  | 2048            | 2048             |

In [4]:
import torch.nn as nn
import copy
from modules.model_trainer import modelTrainer
from modules.evaluate_model import ModelEvaluator
from pwkd import PWKDLoss, make_aux_fn, finalise_student

RESNET50_CHANNELS = {'layer2': 512, 'layer3': 1024, 'layer4': 2048}

PRUNING_RATIOS = [0.10, 0.25, 0.50, 0.70, 0.90]
NUM_EPOCHS     = 10
LAM            = 0.2
KD_TEMP        = 4.0
SPARSE_WEIGHT  = 1e-4

evaluator = ModelEvaluator(
    data_loader=data_prep.val_loader,
    class_names=data_prep.class_names,
    device=str(device),
)

teacher = copy.deepcopy(resnet50_pretrained).eval()
for p in teacher.parameters():
    p.requires_grad = False

pwkd_metrics = []

print(f'ResNet50 baseline F1:    {baseline_metrics["f1_macro"]:.4f}')
print(f'ResNet50 baseline params: {BASELINE_PARAMS:,}')


for ratio in PRUNING_RATIOS:
    label = f'pwkd_self_r50_r{int(ratio * 100)}'
    print(f'\n{"="*60}')
    print(f'  PWKD Option 2 (ResNet50) — pruning ratio {ratio:.0%}')
    print(f'{"="*60}')

    student = copy.deepcopy(resnet50_pretrained).to(device)
    for p in student.parameters():
        p.requires_grad = True

    pwkd_loss = PWKDLoss(
        student          = student,
        teacher          = teacher,
        pruning_ratio    = ratio,
        teacher_channels = RESNET50_CHANNELS,
        student_channels = RESNET50_CHANNELS,
        class_weights    = data_prep.class_weights.to(device)
                           if data_prep.class_weights is not None else None,
        lam              = LAM,
        kd_temp          = KD_TEMP,
        sparse_weight    = SPARSE_WEIGHT,
    ).to(device)

    trainer = modelTrainer(
        model      = student,
        data_prep  = data_prep,
        device     = device,
        learn_rate = 2e-4,
        num_epochs = NUM_EPOCHS,
        model_name = label,
    )
    trainer.loss_fn   = pwkd_loss
    trainer.optimizer = torch.optim.AdamW(
        list(student.parameters()) + list(pwkd_loss.parameters()),
        lr=2e-4, weight_decay=1e-2,
    )
    trainer.create_classnum_to_label_map(data_prep.class_names)
    trainer.train_all(save_as_object=True, aux_forward_fn=make_aux_fn(teacher), pwkd=True)

    student = finalise_student(student, pwkd_loss, data_prep.train_loader, device)

    import os
    save_dir = os.path.join('trained_models', label)
    os.makedirs(save_dir, exist_ok=True)
    torch.save({'model': student, 'epoch': NUM_EPOCHS},
               os.path.join(save_dir, f'{label}_full.pth'))
    print(f'Saved → {save_dir}/{label}_full.pth')

    student.eval()
    metrics = evaluator.evaluate_single(student, label)

    pwkd_metrics.append({
        'Pruning Ratio':      f'{int(ratio*100)}%',
        'Size Reduction (%)': round((BASELINE_PARAMS - metrics['total_parameters'])
                                    / BASELINE_PARAMS * 100, 2),
        'F1 Score':           round(metrics['f1_macro'],     4),
        'Size (MB)':          round(metrics['model_size_mb'], 1),
        'Latency (ms)':       round(metrics['avg_latency_ms'], 2),
    })
    print(f'  ratio: {ratio:.0%} | F1: {metrics["f1_macro"]:.4f} | '
          f'params: {metrics["total_parameters"]:,} | '
          f'latency: {metrics["avg_latency_ms"]:.2f}ms')

summary_df = pd.DataFrame(pwkd_metrics).set_index('Pruning Ratio')
print('\nPWKD Option 2 (ResNet50) — Results')
display(summary_df)


ResNet50 baseline F1:    0.4867
ResNet50 baseline params: 23,633,021

  PWKD Option 2 (ResNet50) — pruning ratio 10%
  Epoch 1/10 [  4.8%]  loss: 1.6986
  Epoch 1/10 [  9.6%]  loss: 1.8080
  Epoch 1/10 [ 14.5%]  loss: 1.6390
  Epoch 1/10 [ 19.3%]  loss: 1.7549
  Epoch 1/10 [ 24.1%]  loss: 1.6983
  Epoch 1/10 [ 28.9%]  loss: 1.5767
  Epoch 1/10 [ 33.7%]  loss: 1.3698
  Epoch 1/10 [ 38.6%]  loss: 1.3338
  Epoch 1/10 [ 43.4%]  loss: 1.2441
  Epoch 1/10 [ 48.2%]  loss: 1.2156
  Epoch 1/10 [ 53.0%]  loss: 1.0117
  Epoch 1/10 [ 57.8%]  loss: 1.0255
  Epoch 1/10 [ 62.7%]  loss: 0.8561
  Epoch 1/10 [ 67.5%]  loss: 0.9334
  Epoch 1/10 [ 72.3%]  loss: 0.8918
  Epoch 1/10 [ 77.1%]  loss: 0.8055
  Epoch 1/10 [ 81.9%]  loss: 0.7813
  Epoch 1/10 [ 86.7%]  loss: 0.8445
  Epoch 1/10 [ 91.6%]  loss: 0.7270
  Epoch 1/10 [ 96.4%]  loss: 0.7205
  Epoch 1/10 [100.0%]  loss: 0.6381


Validating: 100%|██████████| 50/50 [00:03<00:00, 15.52batch/s]



Epoch 1/10
Train Loss: 1.1766 | Train F1: 0.5508
Val Loss: 1.8624 | Val F1: 0.3392
Epoch Time: 83.09s

  Epoch 2/10 [  4.8%]  loss: 0.6421
  Epoch 2/10 [  9.6%]  loss: 0.7673
  Epoch 2/10 [ 14.5%]  loss: 0.6194
  Epoch 2/10 [ 19.3%]  loss: 0.7054
  Epoch 2/10 [ 24.1%]  loss: 0.6418
  Epoch 2/10 [ 28.9%]  loss: 0.6113
  Epoch 2/10 [ 33.7%]  loss: 0.5786
  Epoch 2/10 [ 38.6%]  loss: 0.6168
  Epoch 2/10 [ 43.4%]  loss: 0.6297
  Epoch 2/10 [ 48.2%]  loss: 0.5810
  Epoch 2/10 [ 53.0%]  loss: 0.5850
  Epoch 2/10 [ 57.8%]  loss: 0.5506
  Epoch 2/10 [ 62.7%]  loss: 0.5951
  Epoch 2/10 [ 67.5%]  loss: 0.5492
  Epoch 2/10 [ 72.3%]  loss: 0.5798
  Epoch 2/10 [ 77.1%]  loss: 0.5027
  Epoch 2/10 [ 81.9%]  loss: 0.5502
  Epoch 2/10 [ 86.7%]  loss: 0.5543
  Epoch 2/10 [ 91.6%]  loss: 0.5796
  Epoch 2/10 [ 96.4%]  loss: 0.5313
  Epoch 2/10 [100.0%]  loss: 0.5922


Validating: 100%|██████████| 50/50 [00:03<00:00, 15.63batch/s]



Epoch 2/10
Train Loss: 0.5983 | Train F1: 0.6944
Val Loss: 1.7453 | Val F1: 0.3756
Epoch Time: 82.45s

  Epoch 3/10 [  4.8%]  loss: 0.5583
  Epoch 3/10 [  9.6%]  loss: 0.5152
  Epoch 3/10 [ 14.5%]  loss: 0.5257
  Epoch 3/10 [ 19.3%]  loss: 0.5228
  Epoch 3/10 [ 24.1%]  loss: 0.4995
  Epoch 3/10 [ 28.9%]  loss: 0.4980
  Epoch 3/10 [ 33.7%]  loss: 0.4892
  Epoch 3/10 [ 38.6%]  loss: 0.4763
  Epoch 3/10 [ 43.4%]  loss: 0.4469
  Epoch 3/10 [ 48.2%]  loss: 0.4479
  Epoch 3/10 [ 53.0%]  loss: 0.4822
  Epoch 3/10 [ 57.8%]  loss: 0.4725
  Epoch 3/10 [ 62.7%]  loss: 0.4319
  Epoch 3/10 [ 67.5%]  loss: 0.4931
  Epoch 3/10 [ 72.3%]  loss: 0.4413
  Epoch 3/10 [ 77.1%]  loss: 0.4430
  Epoch 3/10 [ 81.9%]  loss: 0.4286
  Epoch 3/10 [ 86.7%]  loss: 0.4508
  Epoch 3/10 [ 91.6%]  loss: 0.4837
  Epoch 3/10 [ 96.4%]  loss: 0.4490
  Epoch 3/10 [100.0%]  loss: 0.4456


...


Epoch 9/10
Train Loss: 0.3846 | Train F1: 0.8239
Val Loss: 1.7216 | Val F1: 0.3906
Epoch Time: 82.57s



  Epoch 10/10 [  4.8%]  loss: 0.3982
  Epoch 10/10 [  9.6%]  loss: 0.3564
  Epoch 10/10 [ 14.5%]  loss: 0.3408
  Epoch 10/10 [ 19.3%]  loss: 0.3664
  Epoch 10/10 [ 24.1%]  loss: 0.3722
  Epoch 10/10 [ 28.9%]  loss: 0.3783
  Epoch 10/10 [ 33.7%]  loss: 0.3789
  Epoch 10/10 [ 38.6%]  loss: 0.3754
  Epoch 10/10 [ 43.4%]  loss: 0.3891
  Epoch 10/10 [ 48.2%]  loss: 0.3700
  Epoch 10/10 [ 53.0%]  loss: 0.3513
  Epoch 10/10 [ 57.8%]  loss: 0.3604
  Epoch 10/10 [ 62.7%]  loss: 0.3932
  Epoch 10/10 [ 67.5%]  loss: 0.3916
  Epoch 10/10 [ 72.3%]  loss: 0.3582
  Epoch 10/10 [ 77.1%]  loss: 0.4043
  Epoch 10/10 [ 81.9%]  loss: 0.3748
  Epoch 10/10 [ 86.7%]  loss: 0.3988
  Epoch 10/10 [ 91.6%]  loss: 0.3861
  Epoch 10/10 [ 96.4%]  loss: 0.4014
  Epoch 10/10 [100.0%]  loss: 0.3741


Validating: 100%|██████████| 50/50 [00:03<00:00, 15.56batch/s]



Epoch 10/10
Train Loss: 0.3772 | Train F1: 0.8293
Val Loss: 1.6153 | Val F1: 0.4295
Epoch Time: 82.56s

Finalised: 3391/3776 conv1 channels zeroed (89.8%)
Saved → trained_models/pwkd_self_r50_r90/pwkd_self_r50_r90_full.pth

Warming up pwkd_self_r50_r90...
Running inference...
  ratio: 90% | F1: 0.0001 | params: 9,576,573 | latency: 1.13ms

PWKD Option 2 (ResNet50) — Results


,Size Reduction (%),F1 Score,Size (MB),Latency (ms)
Pruning Ratio,,,,
10%,6.53,0.3489,90.4,1.13
25%,16.55,0.2388,90.4,1.12
50%,33.10,0.0102,90.4,1.12
70%,46.27,0.0019,90.4,1.13
90%,59.48,0.0001,90.4,1.13


In [6]:
# only run this cell if not running the previous cell
import torch.nn as nn
import copy
from modules.model_trainer import modelTrainer
from modules.evaluate_model import ModelEvaluator
from pwkd import PWKDLoss, make_aux_fn, finalise_student

RESNET50_CHANNELS = {'layer2': 512, 'layer3': 1024, 'layer4': 2048}

PRUNING_RATIOS = [0.10, 0.25, 0.50, 0.70, 0.90]
NUM_EPOCHS     = 10
LAM            = 0.2
KD_TEMP        = 4.0
SPARSE_WEIGHT  = 1e-4

evaluator = ModelEvaluator(
    data_loader=data_prep.val_loader,
    class_names=data_prep.class_names,
    device=str(device),
)

teacher = copy.deepcopy(resnet50_pretrained).eval()
for p in teacher.parameters():
    p.requires_grad = False

pwkd_metrics = []

In [7]:
# ── Extra ratios to better capture the Pareto curve descent ──────────────────
# Run this cell independently — existing trained models are not affected.

EXTRA_RATIOS = [0.2, 0.3, 0.35, 0.40, 0.45]   # between the good and collapsed points

for ratio in EXTRA_RATIOS:
    label = f'pwkd_self_r50_r{int(ratio * 100)}'
    print(f'\n{"="*60}')
    print(f'  PWKD Option 2 (ResNet50) — pruning ratio {ratio:.0%}')
    print(f'{"="*60}')

    student = copy.deepcopy(resnet50_pretrained).to(device)
    for p in student.parameters():
        p.requires_grad = True

    pwkd_loss = PWKDLoss(
        student          = student,
        teacher          = teacher,
        pruning_ratio    = ratio,
        teacher_channels = RESNET50_CHANNELS,
        student_channels = RESNET50_CHANNELS,
        class_weights    = data_prep.class_weights.to(device)
                           if data_prep.class_weights is not None else None,
        lam              = LAM,
        kd_temp          = KD_TEMP,
        sparse_weight    = SPARSE_WEIGHT,
    ).to(device)

    trainer = modelTrainer(
        model      = student,
        data_prep  = data_prep,
        device     = device,
        learn_rate = 5e-4,
        num_epochs = NUM_EPOCHS,
        model_name = label,
    )
    trainer.loss_fn   = pwkd_loss
    trainer.optimizer = torch.optim.AdamW(
        list(student.parameters()) + list(pwkd_loss.parameters()),
        lr=5e-4, weight_decay=1e-2,
    )
    trainer.create_classnum_to_label_map(data_prep.class_names)
    trainer.train_all(save_as_object=True, aux_forward_fn=make_aux_fn(teacher), pwkd=True)

    student = finalise_student(student, pwkd_loss, data_prep.train_loader, device)

    save_dir = os.path.join('trained_models', label)
    os.makedirs(save_dir, exist_ok=True)
    torch.save({'model': student, 'epoch': NUM_EPOCHS},
               os.path.join(save_dir, f'{label}_full.pth'))
    print(f'Saved → {save_dir}/{label}_full.pth')

    student.eval()
    metrics = evaluator.evaluate_single(student, label)

    pwkd_metrics.append({
        'Pruning Ratio':      f'{int(ratio*100)}%',
        'Size Reduction (%)': round((BASELINE_PARAMS - metrics['total_parameters'])
                                    / BASELINE_PARAMS * 100, 2),
        'F1 Score':           round(metrics['f1_macro'],     4),
        'Size (MB)':          round(metrics['model_size_mb'], 1),
        'Latency (ms)':       round(metrics['avg_latency_ms'], 2),
    })
    print(f'  ratio: {ratio:.0%} | F1: {metrics["f1_macro"]:.4f} | '
          f'params: {metrics["total_parameters"]:,}')

summary_df = pd.DataFrame(pwkd_metrics).set_index('Pruning Ratio')
display(summary_df)


  PWKD Option 2 (ResNet50) — pruning ratio 20%
  Epoch 1/10 [  4.8%]  loss: 3.6317
  Epoch 1/10 [  9.6%]  loss: 3.5960
  Epoch 1/10 [ 14.5%]  loss: 3.0203
  Epoch 1/10 [ 19.3%]  loss: 2.6702
  Epoch 1/10 [ 24.1%]  loss: 2.6090
  Epoch 1/10 [ 28.9%]  loss: 2.5252
  Epoch 1/10 [ 33.7%]  loss: 2.2256
  Epoch 1/10 [ 38.6%]  loss: 2.1543
  Epoch 1/10 [ 43.4%]  loss: 2.3726
  Epoch 1/10 [ 48.2%]  loss: 2.1686
  Epoch 1/10 [ 53.0%]  loss: 1.9540
  Epoch 1/10 [ 57.8%]  loss: 1.8281
  Epoch 1/10 [ 62.7%]  loss: 1.7994
  Epoch 1/10 [ 67.5%]  loss: 1.8400
  Epoch 1/10 [ 72.3%]  loss: 1.6397
  Epoch 1/10 [ 77.1%]  loss: 1.8097
  Epoch 1/10 [ 81.9%]  loss: 1.6320
  Epoch 1/10 [ 86.7%]  loss: 1.7169
  Epoch 1/10 [ 91.6%]  loss: 1.7350
  Epoch 1/10 [ 96.4%]  loss: 1.5231
  Epoch 1/10 [100.0%]  loss: 1.6263


Validating: 100%|██████████| 50/50 [00:03<00:00, 14.95batch/s]


Epoch 1/10
Train Loss: 2.2010 | Train F1: 0.2951
Val Loss: 2.7077 | Val F1: 0.1768
Epoch Time: 96.60s



  Epoch 2/10 [  4.8%]  loss: 1.7177
  Epoch 2/10 [  9.6%]  loss: 1.5105
  Epoch 2/10 [ 14.5%]  loss: 1.4171
  Epoch 2/10 [ 19.3%]  loss: 1.4433
  Epoch 2/10 [ 24.1%]  loss: 1.2419
  Epoch 2/10 [ 28.9%]  loss: 1.2691
  Epoch 2/10 [ 33.7%]  loss: 1.1385
  Epoch 2/10 [ 38.6%]  loss: 1.2358
  Epoch 2/10 [ 43.4%]  loss: 1.2034
  Epoch 2/10 [ 48.2%]  loss: 1.2803
  Epoch 2/10 [ 53.0%]  loss: 1.2598
  Epoch 2/10 [ 57.8%]  loss: 1.1804
  Epoch 2/10 [ 62.7%]  loss: 1.1559
  Epoch 2/10 [ 67.5%]  loss: 1.1494
  Epoch 2/10 [ 72.3%]  loss: 1.2451
  Epoch 2/10 [ 77.1%]  loss: 1.0964
  Epoch 2/10 [ 81.9%]  loss: 1.1944
  Epoch 2/10 [ 86.7%]  loss: 1.1717
  Epoch 2/10 [ 91.6%]  loss: 1.1339
  Epoch 2/10 [ 96.4%]  loss: 1.0322
  Epoch 2/10 [100.0%]  loss: 0.9967


Validating: 100%|██████████| 50/50 [00:04<00:00, 11.19batch/s]

...

  Epoch 10/10 [  4.8%]  loss: 0.5437
  Epoch 10/10 [  9.6%]  loss: 0.5692
  Epoch 10/10 [ 14.5%]  loss: 0.5383
  Epoch 10/10 [ 19.3%]  loss: 0.4909
  Epoch 10/10 [ 24.1%]  loss: 0.5051
  Epoch 10/10 [ 28.9%]  loss: 0.5370
  Epoch 10/10 [ 33.7%]  loss: 0.5289
  Epoch 10/10 [ 38.6%]  loss: 0.5394
  Epoch 10/10 [ 43.4%]  loss: 0.4940
  Epoch 10/10 [ 48.2%]  loss: 0.5046
  Epoch 10/10 [ 53.0%]  loss: 0.5151
  Epoch 10/10 [ 57.8%]  loss: 0.5146
  Epoch 10/10 [ 62.7%]  loss: 0.5092
  Epoch 10/10 [ 67.5%]  loss: 0.4867
  Epoch 10/10 [ 72.3%]  loss: 0.4902
  Epoch 10/10 [ 77.1%]  loss: 0.5251
  Epoch 10/10 [ 81.9%]  loss: 0.5129
  Epoch 10/10 [ 86.7%]  loss: 0.5250
  Epoch 10/10 [ 91.6%]  loss: 0.5089
  Epoch 10/10 [ 96.4%]  loss: 0.4941
  Epoch 10/10 [100.0%]  loss: 0.5755


Validating: 100%|██████████| 50/50 [00:03<00:00, 15.62batch/s]


Epoch 10/10
Train Loss: 0.5188 | Train F1: 0.7167
Val Loss: 1.9995 | Val F1: 0.3125
Epoch Time: 82.29s



Finalised: 1692/3776 conv1 channels zeroed (44.8%)
Saved → trained_models/pwkd_self_r50_r45/pwkd_self_r50_r45_full.pth

Warming up pwkd_self_r50_r45...
Running inference...
  ratio: 45% | F1: 0.0055 | params: 16,609,149


,Size Reduction (%),F1 Score,Size (MB),Latency (ms)
Pruning Ratio,,,,
20%,13.17,0.2588,90.4,1.14
30%,19.74,0.1981,90.4,1.13
35%,23.08,0.0631,90.4,1.13
40%,26.37,0.0348,90.4,1.12
45%,29.72,0.0055,90.4,1.14
